In [1]:
import requests
import os
import datetime as dt
import json
from google.cloud import storage
from google.cloud import bigquery 
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely import wkt

/opt/anaconda3/envs/geo_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/opt/anaconda3/envs/geo_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.bigquery_storage_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.bigquery_storage_v1 past that date.
  warnings.warn(message, FutureWarning)


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
def pull_bucket_file(bucket_name: str, query_date:str):
    """query_date should be formatted as YYYY-MM-DD"""
    
    GSclient = storage.Client()
    bucket = GSclient.bucket(bucket_name)
    blob = bucket.blob(f'{query_date}_311_api_call')
    text = blob.download_as_text()
    
    return pd.DataFrame(json.loads(text))

def pull_bigquery_table(table_name: str):
    """Careful!! This pulls the ENTIRE table, so watch out!"""
    
    BQclient = bigquery.Client()
    query = f'''SELECT * 
            FROM `neighboorhood-nachos.neighborhood_livability_data.{table_name}`'''
    
    return BQclient.query(query).to_dataframe()

def format_bqtable_to_gdf(df):
    df['geometry'] = df['geometry'].apply(wkt.loads)
    return gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

def format_bucket_to_gdf(df):
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['long'], df['lat']), crs='EPSG:4326')

    mask = (gdf['geometry'].is_empty) & (gdf['address'] != 'Not associated with a specific address')
    if mask.any():
        geocodes = gpd.tools.geocode(gdf.loc[mask, 'address'], provider='arcgis')
        gdf.loc[mask, 'geometry'] = geocodes['geometry']
    return gdf

def add_ids_to_gdf(gdf, neighborhoods, police_districts):
    gdf1 = gpd.sjoin(gdf, police_districts[['police_district_id', 'geometry']], how='left', predicate='within')
    gdf1.drop(columns=['index_right'], inplace=True)
    gdf2 = gpd.sjoin(gdf1, neighborhoods[['neighborhood_id', 'geometry']], how='left', predicate='within')

    mask = gdf['geometry'].is_empty
    mask_neighbor = gdf2['neighborhood_id'].isna()
    mask_pd = gdf2['police_district_id'].isna()
    if mask.any():
        gdf2.loc[mask, 'police_district_id'] = 404
        gdf2.loc[mask, 'neighborhood_id'] = 404
    if mask_neighbor.any():
        lookup = neighborhoods.set_index('name')['neighborhood_id']
        gdf2.loc[mask_neighbor, 'neighborhood_id'] = gdf2.loc[mask_neighbor, 'neighborhoods_sffind_boundaries'].map(lookup)
    if mask_pd.any():
        lookup = police_districts.set_index('name')['police_district_id']
        gdf2.loc[mask_pd, 'police_district_id'] = gdf2.loc[mask_pd, 'police_district'].str.capitalize().map(lookup)
    
    gdf2['police_district_id'].fillna(404, inplace=True)
    gdf2['neighborhood_id'].fillna(404, inplace=True)

    return gdf2

def format_gdf_final_columns(gdf):
    gdf1 = gdf[['service_request_id', 'neighborhood_id', 'police_district_id', 'requested_datetime',
               'updated_datetime', 'closed_date', 'status_description', 'status_notes', 'agency_responsible', 'service_name',
               'service_subtype', 'service_details', 'supervisor_district', 'lat', 'long', 'geometry', 'source', 'media_url']].copy()
    gdf1.columns = ['311_incident_id', 'neighborhood_id', 'police_district_id', 'requested_datetime',
                   'updated_datetime', 'closed_datetime', 'status_description', 'status_notes',
                   'agency_responsible', 'service_name', 'service_subtype', 'service_details', 
                   'supervisor_district', 'lat', 'long', 'geometry', 'source', 'media_url']
    
    gdf1.loc[:, 'supervisor_district'] = gdf1['supervisor_district'].fillna(404)
    gdf1['supervisor_district'] = gdf1['supervisor_district'].astype(float)

    timestamp_cols = ['requested_datetime', 'updated_datetime', 'closed_datetime']
    for col in timestamp_cols:
        gdf1[col] = pd.to_datetime(gdf1[col]).dt.tz_localize('UTC')
    
    gdf1 = gdf1.drop_duplicates(subset=['311_incident_id'], keep='last')
    
    gdf1 = gdf1.astype(
    {'311_incident_id': 'int64',
     'neighborhood_id': 'int64',
     'police_district_id': 'int64',
     'status_description': str,
     'status_notes': str,
     'agency_responsible': str,
     'service_name': str,
     'service_subtype': str,
     'service_details': str,
     'supervisor_district': 'int64',
     'lat': float,
     'long': float,
     'geometry': str,
     'source': str,
     'media_url': str})

    return gdf1

def upload_gdf_to_table(gdf, table_name):
    BQclient = bigquery.Client()
    job_config = bigquery.LoadJobConfig(write_disposition='WRITE_APPEND')
    job = BQclient.load_table_from_dataframe(gdf, table_name, job_config=job_config)
    
    return job.result()

In [16]:
bucket_name = os.environ.get("GCS_BUCKET_NAME")
query_date = '2026-06-09'
table_name = 'neighborhood_livability_data.311_incidents'

df = pull_bucket_file(bucket_name, query_date)

df_neighborhoods = pull_bigquery_table('neighborhoods')
gdf_neighborhoods = format_bqtable_to_gdf(df_neighborhoods)
gdf_neighborhoods.columns = ['neighborhood_id', 'name', 'geometry']

df_policedistricts = pull_bigquery_table('police_districts')
gdf_policedistricts = format_bqtable_to_gdf(df_policedistricts)
gdf_policedistricts.columns = ['police_district_id', 'name', 'geometry']

gdf = format_bucket_to_gdf(df)
gdf = add_ids_to_gdf(gdf, gdf_neighborhoods, gdf_policedistricts)
gdf = format_gdf_final_columns(gdf)

In [17]:
BQclient = bigquery.Client()
job_config = bigquery.LoadJobConfig(write_disposition='WRITE_APPEND')
job = BQclient.load_table_from_dataframe(gdf, table_name, job_config=job_config)
job.result()

LoadJob<project=neighboorhood-nachos, location=us-central1, id=33d7ceae-8ef8-40cd-be2f-d3c759c3de02>

In [18]:
bucket_name = os.environ.get("GCS_BUCKET_NAME")
table_name = '311_incidents'
staging_table = '311_incidents__stage'
final_cols = ['311_incident_id', 'neighborhood_id', 'police_district_id', 'requested_datetime',
               'updated_datetime', 'closed_datetime', 'status_description', 'status_notes',
               'agency_responsible', 'service_name', 'service_subtype', 'service_details', 
               'supervisor_district', 'lat', 'long', 'geometry', 'source', 'media_url']

backtick_cols = [f"`{col}`" for col in final_cols]
update_set = ",".join([f'T.`{col}` = S.`{col}`' for col in final_cols[1:]])
insert_cols = ",".join(backtick_cols)
insert_values = ",".join([f'S.`{col}`' for col in final_cols])

def load_staging(gdf, staging_table):
    table = f'neighboorhood-nachos.neighborhood_livability_data.{staging_table}'
    BQclient = bigquery.Client()
    job_config = bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')
    job = BQclient.load_table_from_dataframe(gdf, table, job_config=job_config)
    return job.result()

def merge_staging_and_real(staging_table, table_name):
    merge_query = f'''
                MERGE `neighboorhood-nachos.neighborhood_livability_data.{table_name}` T
                USING `neighboorhood-nachos.neighborhood_livability_data.{staging_table}` S
                ON T.311_incident_id = S.311_incident_id
                WHEN MATCHED THEN
                    UPDATE SET {update_set}
                WHEN NOT MATCHED THEN
                    INSERT ({insert_cols})
                    VALUES ({insert_values})
                   '''
    query_job = BQclient.query(merge_query)

    return query_job.result()

def set_up(bucket_name, query_date):
    df = pull_bucket_file(bucket_name, query_date)

    df_neighborhoods = pull_bigquery_table('neighborhoods')
    gdf_neighborhoods = format_bqtable_to_gdf(df_neighborhoods)
    gdf_neighborhoods.columns = ['neighborhood_id', 'name', 'geometry']

    df_policedistricts = pull_bigquery_table('police_districts')
    gdf_policedistricts = format_bqtable_to_gdf(df_policedistricts)
    gdf_policedistricts.columns = ['police_district_id', 'name', 'geometry']

    gdf = format_bucket_to_gdf(df)
    gdf = add_ids_to_gdf(gdf, gdf_neighborhoods, gdf_policedistricts)
    gdf = format_gdf_final_columns(gdf)

    return gdf

In [19]:
query_date = '2026-06-10'
df = pull_bucket_file(bucket_name, query_date)

df_neighborhoods = pull_bigquery_table('neighborhoods')
gdf_neighborhoods = format_bqtable_to_gdf(df_neighborhoods)
gdf_neighborhoods.columns = ['neighborhood_id', 'name', 'geometry']

df_policedistricts = pull_bigquery_table('police_districts')
gdf_policedistricts = format_bqtable_to_gdf(df_policedistricts)
gdf_policedistricts.columns = ['police_district_id', 'name', 'geometry']

gdf = format_bucket_to_gdf(df)
gdf = add_ids_to_gdf(gdf, gdf_neighborhoods, gdf_policedistricts)
gdf = format_gdf_final_columns(gdf)

gdf.head()


,311_incident_id,neighborhood_id,police_district_id,requested_datetime,updated_datetime,closed_datetime,status_description,status_notes,agency_responsible,service_name,service_subtype,service_details,supervisor_district,lat,long,geometry,source,media_url
0,101004004803,113,7,2026-05-06 12:00:24+00:00,2026-06-10 00:00:27+00:00,2026-06-10 00:00:27+00:00,Closed,Case is Invalid - Case Denied - Denied due to ...,MTA - Temporary Sign Request Printing,Temporary Sign Applications,moving_residential_property,nan,404,37.766486,-122.437702,POINT (-122.437702 37.766486),Phone,nan
1,101004220820,84,9,2026-06-09 18:49:52+00:00,2026-06-10 00:31:21+00:00,2026-06-10 01:31:17+00:00,Closed,Case Resolved - Officer responded to request u...,MTA - Parking Enforcement Dispatch,Parking Enforcement,parking_on_sidewalk,nan,8,37.747759,-122.423980,POINT (-122.42398 37.747759),Mobile,{'url': 'https://spot-sf-res.cloudinary.com/im...
2,101004220826,52,3,2026-06-09 18:50:21+00:00,2026-06-10 00:32:00+00:00,2026-06-10 01:31:56+00:00,Closed,Case Resolved - Officer responded to request u...,MTA - Parking Enforcement Dispatch,Parking Enforcement,parking_on_sidewalk,nan,8,37.747921,-122.425343,POINT (-122.425343 37.747921),Mobile,{'url': 'https://spot-sf-res.cloudinary.com/im...
3,101004221589,40,10,2026-06-09 22:58:57+00:00,2026-06-10 00:47:21+00:00,NaT,Open,accepted,PW - Urban Forestry,Sewer,water_leak,water_main,4,37.738496,-122.494182,POINT (-122.494182 37.738496),Phone,nan
4,101004221687,53,3,2026-06-10 00:21:37+00:00,2026-06-10 00:47:23+00:00,NaT,Open,in progress,PW - Street and Environmental Services,Street and Sidewalk Cleaning,garbage_and_debris,human_waste_or_urine,9,37.765194,-122.420943,POINT (-122.420943 37.765194),Mobile,{'url': 'https://spot-sf-res.cloudinary.com/im...


In [20]:
load_staging(gdf, staging_table)

LoadJob<project=neighboorhood-nachos, location=us-central1, id=62672c17-b7d7-44e8-903f-fd8f904532cd>

In [21]:
merge_staging_and_real(staging_table, table_name)

In [22]:
query_date = '2026-06-11'
df = pull_bucket_file(bucket_name, query_date)

gdf = format_bucket_to_gdf(df)
gdf = add_ids_to_gdf(gdf, gdf_neighborhoods, gdf_policedistricts)
gdf = format_gdf_final_columns(gdf)

load_staging(gdf, staging_table)
merge_staging_and_real(staging_table, table_name)

In [23]:
query_date = '2026-06-12'
df = pull_bucket_file(bucket_name, query_date)

gdf = format_bucket_to_gdf(df)
gdf = add_ids_to_gdf(gdf, gdf_neighborhoods, gdf_policedistricts)
gdf = format_gdf_final_columns(gdf)

load_staging(gdf, staging_table)
merge_staging_and_real(staging_table, table_name)

In [28]:
for i in range(13,24):
    query_date = f'2026-06-{i}'
    df = pull_bucket_file(bucket_name, query_date)

    gdf = format_bucket_to_gdf(df)
    gdf = add_ids_to_gdf(gdf, gdf_neighborhoods, gdf_policedistricts)
    gdf = format_gdf_final_columns(gdf)

    load_staging(gdf, staging_table)
    merge_staging_and_real(staging_table, table_name)
    print(f'success for June {i}')

success for June 13
success for June 14
success for June 15
success for June 16
success for June 17
success for June 18
success for June 19
success for June 20
success for June 21
success for June 22
success for June 23


In [29]:
query_date = '2026-06-24'
df = pull_bucket_file(bucket_name, query_date)

gdf = format_bucket_to_gdf(df)
gdf = add_ids_to_gdf(gdf, gdf_neighborhoods, gdf_policedistricts)
gdf = format_gdf_final_columns(gdf)

load_staging(gdf, staging_table)
merge_staging_and_real(staging_table, table_name)